# Playground S5E11 - Predicting Loan Payback

The goal is to predict whether a borrower will successfully repay their loan. The data was generated for the [playground series](https://www.kaggle.com/competitions/playground-series-s5e11/data). Moreover, the [original dataset](https://www.kaggle.com/datasets/nabihazahid/loan-prediction-dataset-2025) [nabiha zahid](https://www.kaggle.com/nabihazahid) was also synthetically generated using Python libraries such as Faker, NumPy, and Pandas, created solely for educational and research use.


### Cite
Yao Yan, Walter Reade, Elizabeth Park. Predicting Loan Payback. https://kaggle.com/competitions/playground-series-s5e11, 2025. Kaggle.

## Imports

In [1]:
#pip install flaml --quiet

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
    return

## About the Data
| Category                  | Field               | Type      | Description                                              |
|---------------------------|---------------------|-----------|----------------------------------------------------------|
| **Borrower’s Demographics** | gender              | category  | Borrower's gender (Male/Female).                         |
|                           | marital_status      | category  | Marital status (Single, Married, Divorced).              |
|                           | education_level     | category  | Education level (High School, Bachelor, Master, PhD).    |
| **Financial Information**  | annual_income       | float64   | Borrower's yearly income.                                |
|                           | debt_to_income_ratio| float64   | Ratio of borrower’s debt to their income. Lower = better.|
|                           | credit_score        | int64     | Credit bureau score (e.g., FICO). Higher = less risky.   |
| **Employment Information** | employment_status   | category  | Current employment type (Employed, Self-Employed, Unemployed). |
| **Loan Information**       | loan_amount         | float64   | Amount of loan taken.                                    |
|                           | loan_purpose        | category  | Loan purpose (Car, Education, Home, Medical, etc.).     |
|                           | interest_rate       | float
| **Target Variable**       | loan_paid_back         | int64       | Target variable:<br>1 → Borrower paid loan in full.<br>0 → Borrower defaulted (did not repay fully).|

In [3]:
train = pd.read_csv('/kaggle/input/playground-series-s5e11/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s5e11/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s5e11/sample_submission.csv')

In [4]:
def downcasting(data: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Downcast numerical columns in the DataFrame to reduce memory usage.
    
    Parameters:
        data (pd.DataFrame): Input DataFrame to optimize.
        verbose (bool): If True, prints memory usage before and after optimization.
    
    Returns:
        pd.DataFrame: Optimized DataFrame with downcasted numeric columns.
    """
    mem_before = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {mem_before:.2f} MB")
    
    for col in data.select_dtypes(include=["number"]).columns:
        if pd.api.types.is_integer_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="integer")
        elif pd.api.types.is_float_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="float")
    
    mem_after = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {mem_after:.2f} MB")
        print(f"Decreased by {(100 * (mem_before - mem_after) / mem_before):.1f}%\n")
    
    return data

print("Train train:")
train = downcasting(train)
print("Test train:")
test = downcasting(test)

Train train:
Memory usage of dataframe is 58.91 MB
Memory usage after optimization is: 46.45 MB
Decreased by 21.2%

Test train:
Memory usage of dataframe is 23.31 MB
Memory usage after optimization is: 18.94 MB
Decreased by 18.7%



In [5]:
target = train["loan_paid_back"]
cols = train.drop(columns=["id", "loan_paid_back"]).columns.tolist()

# Categorical features
cat_cols = [c for c in cols if train[c].dtype in ["object","category"]]
# Numerical features
num_cols = [c for c in cols if train[c].dtype not in ["object","category","bool"]]

print("Categorical cols:")
print(cat_cols)
print("Numerical cols:")
print(num_cols)

Categorical cols:
['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Numerical cols:
['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


In [6]:
for col in cat_cols:
    unique_values = train[col].dropna().unique()  # or orig[col] depending on your dataset
    print(f"Unique values in {col}: {unique_values}")

Unique values in gender: ['Female' 'Male' 'Other']
Unique values in marital_status: ['Single' 'Married' 'Divorced' 'Widowed']
Unique values in education_level: ['High School' "Master's" "Bachelor's" 'PhD' 'Other']
Unique values in employment_status: ['Self-employed' 'Employed' 'Unemployed' 'Retired' 'Student']
Unique values in loan_purpose: ['Other' 'Debt consolidation' 'Home' 'Education' 'Vacation' 'Car'
 'Medical' 'Business']
Unique values in grade_subgrade: ['C3' 'D3' 'C5' 'F1' 'D1' 'D5' 'C2' 'C1' 'F5' 'D4' 'C4' 'D2' 'E5' 'B1'
 'B2' 'F4' 'A4' 'E1' 'F2' 'B4' 'E4' 'B3' 'E3' 'B5' 'E2' 'F3' 'A5' 'A3'
 'A1' 'A2']


## Feature Engineering

### One-Hot Encoding (Nominal Categories)

Used for variables where categories have **no natural order**. This encoding creates binary columns for each category, preserving the distinctiveness without implying any hierarchy.

- **gender**  
  Unique values: ['Female', 'Male', 'Other']  
  Reason: Nominal, no inherent ranking between these categories.
  
- **marital_status**  
  Unique values: ['Single', 'Married', 'Divorced', 'Widowed']  
  Reason: Nominal, categories are distinct with no ordered relationship.
  
- **employment_status**  
  Unique values: ['Self-employed', 'Employed', 'Unemployed', 'Retired', 'Student']  
  Reason: Different employment types with no ordinal ranking.
  
- **loan_purpose**  
  Unique values: ['Other', 'Debt consolidation', 'Home', 'Education', 'Vacation', 'Car', 'Medical', 'Business']  
  Reason: Diverse reasons for loans with no natural sequence.

---

### Ordinal Encoding (Ordered Categories)

Used when categories have a **clear, meaningful order**. This encoding maps categories to integers representing their rank.

- **education_level**  
  Unique values: ['High School', "Master's", "Bachelor's", 'PhD', 'Other']  
  Reason: Higher education levels imply greater academic attainment. 
  Other < High School < Bachelor's < Master's < PhD.

- **grade_subgrade**  
  Unique values: ['C3', 'D3', 'C5', 'F1', 'D1', 'D5', 'C2', 'C1', 'F5', 'D4', 'C4', 'D2', 'E5', 'B1', 'B2', 'F4', 'A4', 'E1', 'F2', 'B4', 'E4', 'B3', 'E3', 'B5', 'E2', 'F3', 'A5', 'A3', 'A1', 'A2']  
  Reason: This represents loan risk categories with an order from 'A' (best) down to 'F' (worst). Map grades to ordinal integers reflecting this risk hierarchy, e.g., A1=1,

### Numerical Feature Transformations
Applied to continuous variables, especially those with skewed distributions, to improve model performance and reduce effect of outliers.

Variables: annual_income, debt_to_income_ratio, credit_score, loan_amount, interest_rate

Skewness Checking:
Calculate skewness for each numerical column. Variables with heavy right skew (skewness > 1) are candidates for transformation.

Log Transformation:
Apply log transform (e.g., log1p) to right-skewed variables to normalize their distribution. This helps reduce the influence of extreme values.

Scaling:
After transformation, apply standard scaling (zero mean, unit variance) to all numerical features to ensure comparability and improve model convergence.

In [7]:
def encode_categorical(df):
    # Ordinal mapping for education_level
    education_order = ['Other', 'High School', "Bachelor's", "Master's", 'PhD']
    education_map = {level: i for i, level in enumerate(education_order)}
    
    # Ordinal mapping for grade_subgrade based on risk (A best, F worst)
    grade_order = [
        'A1','A2','A3','A4','A5',
        'B1','B2','B3','B4','B5',
        'C1','C2','C3','C4','C5',
        'D1','D2','D3','D4','D5',
        'E1','E2','E3','E4','E5',
        'F1','F2','F3','F4','F5'
    ]
    grade_map = {grade: i+1 for i, grade in enumerate(grade_order)}
    
    # Apply ordinal encoding
    df['education_level_encoded'] = df['education_level'].map(education_map)
    df['grade_subgrade_encoded'] = df['grade_subgrade'].map(grade_map)
    
    # Drop original ordinal columns
    df = df.drop(['education_level', 'grade_subgrade'], axis=1)
    
    # One-hot encode nominal categorical columns
    nominal_cols = ['gender', 'marital_status', 'employment_status', 'loan_purpose']
    df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
    
    return df

def scale_numerical(df, numerical_columns=num_cols):
    df_copy = df.copy()
    
    # Calculate skewness for numerical columns
    skewness = df_copy[numerical_columns].skew()
    
    # Columns to log-transform (right skew > 1)
    skewed_columns = skewness[skewness > 1].index
    for col in skewed_columns:
        df_copy[col] = np.log1p(df_copy[col])
    
    # Scale numerical columns using StandardScaler
    scaler = StandardScaler()
    df_copy[numerical_columns] = scaler.fit_transform(df_copy[numerical_columns])
    
    return df_copy

def feature_engineering_pipeline(df):
    df = df.drop(columns=["id", "loan_paid_back"], errors='ignore')
    df = encode_categorical(df)
    df = scale_numerical(df)
    return df

train = feature_engineering_pipeline(train)

In [8]:
X = train
y = target.astype(int)
# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
%%skip
# Initialize AutoML with ROC AUC metric
automl = AutoML()

settings = {
    "time_budget": 300,        # in seconds
    "metric": 'roc_auc',       # Use ROC AUC
    "task": 'classification',  # Task type
    "log_file_name": "flaml.log",
}

# Train AutoML model
automl.fit(X_train=X_train, y_train=y_train, **settings)

# Predict probabilities for ROC AUC calculation
y_pred_proba = automl.predict_proba(X_test)[:, 1]

# Calculate ROC AUC score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")

best_model = automl.model.estimator

# Print some information about the best model
print(f"Best learner: {automl.best_estimator}")
print(f"Best model estimator object: {best_model}")
print(f"Best validation loss (lower is better): {automl.best_loss}")

In [10]:
%%skip
LGBMClassifier(colsample_bytree=0.2150302782395718,
               learning_rate=0.06530127880393859, max_bin=1023,
               min_child_samples=13, n_estimators=1265, n_jobs=-1,
               num_leaves=15, reg_alpha=0.0030413642053562034,
               reg_lambda=0.03444496925683231, verbose=-1)

## Predict

In [11]:
test = feature_engineering_pipeline(test)

# Initialize the model with your parameters
model = lgb.LGBMClassifier(
    colsample_bytree=0.7610534336273627,
    learning_rate=0.15662398373030859,
    max_bin=1023,
    min_child_samples=7,
    n_estimators=53,
    n_jobs=-1,
    num_leaves=4,
    reg_alpha=0.0009765625,
    reg_lambda=0.006425898219455277,
    verbose=-1
)

model.fit(X, y)

y_pred = model.predict(test)
y_pred_proba = model.predict_proba(test)[:, 1]

In [14]:
submission['loan_paid_back'] = y_pred_proba
submission.to_csv('submission.csv', index=False)
submission.head()

,id,loan_paid_back
0,593994,0.912720
1,593995,0.979988
2,593996,0.455453
3,593997,0.932503
4,593998,0.945920
